# AQI Feature Engineering

This notebook creates time-based and historical features from the processed AQI dataset.

The engineered features are designed for time-series forecasting while maintaining the chronological structure of the observations.

The workflow includes:
- Time-based features
- Lag features
- AQI change
- Rolling AQI average
- Data validation
- Saving the feature-engineered dataset

In [1]:
import numpy as np
import pandas as pd

## 1. Load Processed AQI Data

The cleaned historical AQI dataset generated by the preprocessing stage is loaded for feature engineering.

In [2]:
df = pd.read_csv("../data/processed/aqi_processed_data.csv")

df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")

df = (
    df
    .dropna(subset=["datetime"])
    .sort_values("datetime")
    .reset_index(drop=True)
)

print("Dataset shape:", df.shape)
print("Date range:", df["datetime"].min(), "to", df["datetime"].max())

display(df.head())

Dataset shape: (120, 10)
Date range: 2026-08-30 03:00:00 to 2026-09-04 02:00:00


,datetime,AQI,CO,NO,NO2,O3,SO2,PM2_5,PM10,NH3
0,2026-08-30 03:00:00,2,70.64,0.05,1.18,35.02,0.90,6.99,20.53,0.00
1,2026-08-30 04:00:00,1,71.37,0.13,1.16,36.15,0.96,6.50,18.09,0.00
2,2026-08-30 05:00:00,1,71.14,0.17,0.98,38.45,0.94,6.11,15.69,0.00
3,2026-08-30 06:00:00,1,70.22,0.16,0.77,41.43,0.91,6.07,14.75,0.01
4,2026-08-30 07:00:00,1,70.41,0.11,0.71,43.64,0.87,6.01,15.01,0.00


## 2. Create Time-Based Features

Time-based features capture recurring temporal patterns that may influence air quality.

The hour, day, month, and weekday are extracted from the observation timestamp.

In [3]:
df["hour"] = df["datetime"].dt.hour
df["day"] = df["datetime"].dt.day
df["month"] = df["datetime"].dt.month
df["weekday"] = df["datetime"].dt.weekday

print("Time-based features created.")

display(
    df[
        ["datetime", "hour", "day", "month", "weekday"]
    ].head()
)

Time-based features created.


,datetime,hour,day,month,weekday
0,2026-08-30 03:00:00,3,30,8,6
1,2026-08-30 04:00:00,4,30,8,6
2,2026-08-30 05:00:00,5,30,8,6
3,2026-08-30 06:00:00,6,30,8,6
4,2026-08-30 07:00:00,7,30,8,6


## 3. Create AQI Lag Feature

The lag feature represents the previous observed AQI value.

It provides historical AQI information to the forecasting model without using future observations.

In [5]:
df["AQI_lag_1"] = df["AQI"].shift(1)



In [6]:
print("AQI lag feature created.")

AQI lag feature created.


## 4. Create AQI Change Feature

AQI change measures the difference between the current AQI and the previous observation.

The first observation has no previous AQI value, so its change is initialized to zero.

In [7]:
df["AQI_change"] = df["AQI"] - df["AQI_lag_1"]

df["AQI_change"] = df["AQI_change"].fillna(0)

print("AQI change feature created.")

AQI change feature created.


## 5. Create Rolling AQI Feature

A short-term rolling average is calculated from previous AQI observations.

The current AQI is excluded from the rolling calculation to reduce information leakage when the feature is later used for forecasting.

In [8]:
df["AQI_rolling_avg"] = (
    df["AQI"]
    .shift(1)
    .rolling(window=3, min_periods=1)
    .mean()
)

df["AQI_rolling_avg"] = df["AQI_rolling_avg"].bfill()

print("Rolling AQI feature created.")

Rolling AQI feature created.


## 6. Feature Engineering Validation

The engineered dataset is checked for dimensions, missing values, duplicate timestamps, and feature availability.

In [9]:
print("Final dataframe shape:", df.shape)

print("\nFinal columns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate timestamps:")
print(df["datetime"].duplicated().sum())

print("\nDate range:")
print(df["datetime"].min())
print(df["datetime"].max())

display(df.head())
display(df.tail())

Final dataframe shape: (120, 17)

Final columns:
['datetime', 'AQI', 'CO', 'NO', 'NO2', 'O3', 'SO2', 'PM2_5', 'PM10', 'NH3', 'hour', 'day', 'month', 'weekday', 'AQI_lag_1', 'AQI_change', 'AQI_rolling_avg']

Missing values:
datetime           0
AQI                0
CO                 0
NO                 0
NO2                0
O3                 0
SO2                0
PM2_5              0
PM10               0
NH3                0
hour               0
day                0
month              0
weekday            0
AQI_lag_1          1
AQI_change         0
AQI_rolling_avg    0
dtype: int64

Duplicate timestamps:
0

Date range:
2026-08-30 03:00:00
2026-09-04 02:00:00


,datetime,AQI,CO,NO,NO2,O3,SO2,PM2_5,PM10,NH3,hour,day,month,weekday,AQI_lag_1,AQI_change,AQI_rolling_avg
0,2026-08-30 03:00:00,2,70.64,0.05,1.18,35.02,0.90,6.99,20.53,0.00,3,30,8,6,NaN,0.0,2.000000
1,2026-08-30 04:00:00,1,71.37,0.13,1.16,36.15,0.96,6.50,18.09,0.00,4,30,8,6,2.0,-1.0,2.000000
2,2026-08-30 05:00:00,1,71.14,0.17,0.98,38.45,0.94,6.11,15.69,0.00,5,30,8,6,1.0,0.0,1.500000
3,2026-08-30 06:00:00,1,70.22,0.16,0.77,41.43,0.91,6.07,14.75,0.01,6,30,8,6,1.0,0.0,1.333333
4,2026-08-30 07:00:00,1,70.41,0.11,0.71,43.64,0.87,6.01,15.01,0.00,7,30,8,6,1.0,0.0,1.000000


,datetime,AQI,CO,NO,NO2,O3,SO2,PM2_5,PM10,NH3,hour,day,month,weekday,AQI_lag_1,AQI_change,AQI_rolling_avg
115,2026-09-03 22:00:00,1,69.87,0.00,0.96,34.62,0.78,5.28,13.57,0.0,22,3,9,3,1.0,0.0,1.0
116,2026-09-03 23:00:00,1,70.22,0.00,0.89,35.06,0.74,5.23,13.99,0.0,23,3,9,3,1.0,0.0,1.0
117,2026-09-04 00:00:00,1,70.86,0.00,0.89,35.42,0.74,5.20,14.37,0.0,0,4,9,4,1.0,0.0,1.0
118,2026-09-04 01:00:00,1,71.96,0.00,1.02,35.57,0.78,5.18,14.70,0.0,1,4,9,4,1.0,0.0,1.0
119,2026-09-04 02:00:00,1,74.20,0.01,1.39,35.42,0.91,5.16,14.89,0.0,2,4,9,4,1.0,0.0,1.0


## 7. Save Feature-Engineered Dataset

The completed feature-engineered dataset is saved for exploratory analysis and machine learning model development.

In [10]:
import os

os.makedirs("../data/processed", exist_ok=True)

output_path = "../data/processed/AQI_feature_engineered.csv"

df.to_csv(output_path, index=False)

print("Feature-engineered dataset saved successfully.")
print("Path:", output_path)
print("Final shape:", df.shape)

Feature-engineered dataset saved successfully.
Path: ../data/processed/AQI_feature_engineered.csv
Final shape: (120, 17)


## Conclusion

The historical AQI dataset has been enriched with temporal and lag-based features.

The resulting dataset is ready for exploratory data analysis and time-series forecasting model development.